In [1]:
!pip install chromadb sentence-transformers groq pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

In [2]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os

In [3]:
df = pd.read_csv("college_notes.csv")

In [4]:
chroma_client = chromadb.Client()

smart_notes = chroma_client.get_or_create_collection(
    name="college_notes"
)

In [5]:
documents = df["content"].tolist()

ids = [f"note_{i}" for i in range(len(documents))]

In [6]:
smart_notes.add(
    documents=documents,
    ids=ids
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 40.8MiB/s]


In [7]:
results = smart_notes.query(
    query_texts=["What is ETL?"],
    n_results=3
)

In [8]:
print("Retrieved Notes:")
print("=" * 50)

for doc in results["documents"][0]:
    print(doc)
    print("-" * 50)

Retrieved Notes:
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.
--------------------------------------------------
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.
--------------------------------------------------
RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant documents from a knowledge base and then generates an answer based on those retrieved documents. This reduces hallucination and allows AI to answer questions about specific data.
--------------------------------------------------


In [16]:
from groq import Groq
import os

os.environ["GROQ_API_KEY"] = ""
client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

MODEL = "llama-3.1-8b-instant"

print("Groq Ready")

Groq Ready


In [17]:
def ask_college_assistant(question):

    results = smart_notes.query(
        query_texts=[question],
        n_results=3
    )

    retrieved_docs = "\n".join(
        results["documents"][0]
    )

    prompt = f"""
Use the following college notes to answer.

Notes:
{retrieved_docs}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [18]:
print(client)
print(MODEL)

llama-3.1-8b-instant


In [19]:
try:
    question = "What is ETL?"
    answer = ask_college_assistant(question)
    print(answer)

except Exception as e:
    print("ERROR:")
    print(e)

ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources, transforming it into a clean and structured format, and loading it into a database or data warehouse for analysis.


In [20]:
question = "What is API?"

answer = ask_college_assistant(question)

print(answer)

An API, or Application Programming Interface, is a mechanism that allows two software applications to communicate with each other. In data engineering, APIs are used to retrieve data from external services, such as weather data, stock prices, or social media feeds.


In [21]:
question = "What is RAG?"

answer = ask_college_assistant(question)

print(answer)

RAG stands for Retrieval Augmented Generation, a technique used in AI models where they first retrieve relevant documents from a knowledge base and then generate an answer based on those retrieved documents.


In [22]:
question = "What is ChromaDB?"

answer = ask_college_assistant(question)

print(answer)

Unfortunately, there is no information about ChromaDB in the provided notes. The notes cover data visualization, RAG (Retrieval Augmented Generation), and Pandas, but do not mention ChromaDB.


In [23]:
  question = "What is Data Visualization?"

answer = ask_college_assistant(question)

print(answer)

Data Visualization is the process of representing data as charts, graphs, and visual formats.


In [24]:
question = "What is Machine Learning?"

answer = ask_college_assistant(question)

print(answer)

Based on the provided college notes, Machine Learning is a type of learning where a model learns from labeled data. It involves the model being given input features and correct output labels, and it learns to predict outputs for new unseen inputs.


In [25]:
questions = [
    "What is ETL?",
    "What is API?",
    "What is RAG?",
    "What is Data Visualization?"
]

for q in questions:
    print("="*50)
    print("Question:", q)
    print("Answer:")
    print(ask_college_assistant(q))
    print()

Question: What is ETL?
Answer:
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources, transforming it into a clean and structured format, and loading it into a database or data warehouse for analysis.

Question: What is API?
Answer:
An API, or Application Programming Interface, is a mechanism that allows two software applications to communicate with each other. In data engineering, APIs are used to retrieve data from external services, such as weather data, stock prices, or social media feeds.

Question: What is RAG?
Answer:
RAG stands for Retrieval Augmented Generation, a technique used in AI models where they first retrieve relevant documents from a knowledge base and then generate an answer based on those retrieved documents.

Question: What is Data Visualization?
Answer:
Data Visualization is the process of representing data as charts, graphs, and visual formats.



# Project Conclusion

The College Knowledge Assistant uses Retrieval Augmented Generation (RAG) to answer questions from college notes.

Technologies used:
- Python
- ChromaDB
- Groq API
- Embeddings
- Generative AI

The system retrieves relevant information and generates intelligent responses automatically.